In [26]:
import math
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 5)

In [27]:
def print_table(rows):
    print(f"\n{'n':>3} {'a_n':>12} {'b_n':>12} {'c_n':>12} {'f(c_n)':>14}")
    for n, a, b, c, fc in rows:
        print(f"{n:>3} {a:>12.6f} {b:>12.6f} {c:>12.6f} {fc:>14.6f}")


def plot_function(f, a, b, root, title, filename):
    margin = 0.25 * (b - a)
    xs = np.linspace(a - margin, b + margin, 400)
    ys = [f(x) for x in xs]
    plt.figure()
    plt.axhline(0, color="black", linewidth=0.8)
    plt.plot(xs, ys, label="f(x)")
    plt.plot(root, f(root), "ro", label=f"root ≈ {root:.6f}")
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()


In [28]:
def find_root(func, a, b, eps, next_c, max_iter=200):
    func_a, func_b = func(a), func(b)
    if func_a * func_b > 0:
        raise ValueError("f(a) and f(b) must have opposite signs")

    rows = []
    n = 0
    c = func_c = width = 0

    while n <= max_iter:
        c = next_c(a, b, func_a, func_b)
        func_c = func(c)
        width = b - a
        n += 1
        rows.append((n, a, b, c, func_c))

        if abs(func_c) < eps or width < eps:
            return c, func_c, width, n, rows

        if func_a * func_c < 0:
            b, func_b = c, func_c
        else:
            a, func_a = c, func_c

    return c, func_c, width, n, rows


def bisection(func, a, b, eps, max_iter=200):
    return find_root(func, a, b, eps, lambda a, b, fa, fb: (a + b) / 2, max_iter)


def false_position(func, a, b, eps, max_iter=200):
    return find_root(func, a, b, eps, lambda a, b, fa, fb: (a * fb - b * fa) / (fb - fa), max_iter)


In [29]:
METHODS = {
    "bisection": {
        "solve": bisection,
        "title": "BISECTION METHOD",
        "output_dir": "bisection",
        "tasks": [
            ("e^x - x^2 = 0", lambda x: math.exp(x) - x ** 2, (-2, 0), 0.001),
            ("x^3 - x - 2 = 0", lambda x: x ** 3 - x - 2, (1, 2), 1e-5),
            ("cos(x) - x = 0", lambda x: math.cos(x) - x, (0, 1), 1e-6),
            ("x^3 - 4x - 9 = 0", lambda x: x ** 3 - 4 * x - 9, (2, 3), 1e-5),
            ("e^-x - x = 0", lambda x: math.exp(-x) - x, (0, 1), 1e-6),
            ("x^2 - 5 = 0", lambda x: x ** 2 - 5, (2, 3), 1e-5),
        ],
    },
    "false_position": {
        "solve": false_position,
        "title": "FALSE POSITION METHOD",
        "output_dir": "false-position",
        "tasks": [
            ("x^3 - x - 2 = 0", lambda x: x ** 3 - x - 2, (1, 2), 1e-6),
            ("e^-x - x = 0", lambda x: math.exp(-x) - x, (0, 1), 1e-6),
            ("x^3 - 2x - 5 = 0", lambda x: x ** 3 - 2 * x - 5, (2, 3), 1e-6),
            ("ln(x) + x - 3 = 0", lambda x: math.log(x) + x - 3, (2, 3), 1e-6),
            ("cos(x) - x = 0", lambda x: math.cos(x) - x, (0, 1), 1e-6),
        ],
    },
}


def run_task(method_name, task_num=0):
    method = METHODS[method_name]
    label, func, (a, b), eps = method["tasks"][task_num]

    print("=" * 70)
    print(f"{method['title']}: {label} on [{a}, {b}], eps = {eps}")
    print("=" * 70)

    # a) verify sign change
    func_a, func_b = func(a), func(b)
    print(f"\nf({a}) = {func_a:.6f}")
    print(f"f({b}) = {func_b:.6f}")
    if func_a * func_b < 0:
        print("Sign change confirmed: a root exists in the interval.")
    else:
        raise ValueError("No sign change on the given interval")

    # b)+c) run iterations and record the log
    root, f_root, width, n_iter, rows = method["solve"](func, a, b, eps)
    print_table(rows)

    # d) report final result
    print(f"\nApproximate root: {round(root, 4)}")
    print(f"f(root) = {f_root:.6f}")
    print(f"Total number of iterations: {n_iter}")

    plot_function(
        func, a, b, root,
        f"{method['title'].title()}: {label} on [{a}, {b}]",
        f"{method['output_dir']}/{method_name}_task_{task_num + 1}.png",
    )


In [31]:
user_input = input("Enter task number (0-5): ").strip()
run_task("bisection", int(user_input) if user_input else 0)


BISECTION METHOD: x^3 - x - 2 = 0 on [1, 2], eps = 1e-05

f(1) = -2.000000
f(2) = 4.000000
Sign change confirmed: a root exists in the interval.

  n          a_n          b_n          c_n         f(c_n)
  1     1.000000     2.000000     1.500000      -0.125000
  2     1.500000     2.000000     1.750000       1.609375
  3     1.500000     1.750000     1.625000       0.666016
  4     1.500000     1.625000     1.562500       0.252197
  5     1.500000     1.562500     1.531250       0.059113
  6     1.500000     1.531250     1.515625      -0.034054
  7     1.515625     1.531250     1.523438       0.012250
  8     1.515625     1.523438     1.519531      -0.010971
  9     1.519531     1.523438     1.521484       0.000622
 10     1.519531     1.521484     1.520508      -0.005179
 11     1.520508     1.521484     1.520996      -0.002279
 12     1.520996     1.521484     1.521240      -0.000829
 13     1.521240     1.521484     1.521362      -0.000103
 14     1.521362     1.521484     1.521423

In [32]:
user_input = input("Enter task number (0-4): ").strip()
run_task("false_position", int(user_input) if user_input else 0)


FALSE POSITION METHOD: x^3 - x - 2 = 0 on [1, 2], eps = 1e-06

f(1) = -2.000000
f(2) = 4.000000
Sign change confirmed: a root exists in the interval.

  n          a_n          b_n          c_n         f(c_n)
  1     1.000000     2.000000     1.333333      -0.962963
  2     1.333333     2.000000     1.462687      -0.333339
  3     1.462687     2.000000     1.504019      -0.101818
  4     1.504019     2.000000     1.516331      -0.029895
  5     1.516331     2.000000     1.519919      -0.008675
  6     1.519919     2.000000     1.520957      -0.002509
  7     1.520957     2.000000     1.521258      -0.000725
  8     1.521258     2.000000     1.521344      -0.000209
  9     1.521344     2.000000     1.521370      -0.000060
 10     1.521370     2.000000     1.521377      -0.000017
 11     1.521377     2.000000     1.521379      -0.000005
 12     1.521379     2.000000     1.521379      -0.000001
 13     1.521379     2.000000     1.521380      -0.000000

Approximate root: 1.5214
f(root) = -